# A picture is worth 16 × 16 tokens
### The Hitchhiker's Guide to the Tokenverse, part 3

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/srishtiii28/hitchhiker_tokens/blob/main/notebooks/03_vision_transformers_and_cls.ipynb)

So far we've only dealt with text. But the idea of turning an input into a sequence of manageable units doesn't stop there. What happens when the input is an image?

In this notebook we'll cut an image into patches, turn those patches into embeddings, add positional information, and meet the `[CLS]` token. Then we'll put it all together in a small Vision Transformer and, if you like, train it on a toy dataset.

One thing to keep in mind throughout: patches and patch embeddings are how the input sequence gets built. `[CLS]` is something else, a learned vector that some architectures add to that sequence. It isn't a tokenizer, and it doesn't split anything.

Everything runs on a CPU. A GPU (in Colab: *Runtime → Change runtime type*) only speeds up the optional training at the end.

## 0. Setup

In [ ]:
# Setup: run this cell first.
# In Colab it clones the repository and installs the package. Locally it assumes
# you ran `pip install -e ".[nlp,notebooks,vision,dev]"` from the repo root.
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/srishtiii28/hitchhiker_tokens.git"

if IN_COLAB:
    repo_dir = Path("/content/hitchhiker_tokens")
    if not repo_dir.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", REPO_URL, str(repo_dir)])
    %pip install -q -e "/content/hitchhiker_tokens[nlp,notebooks]"
    # Colab ships with PyTorch; install it only if it is missing.
    if importlib.util.find_spec("torch") is None:
        %pip install -q torch
else:
    repo_dir = Path.cwd()
    while not (repo_dir / "pyproject.toml").exists() and repo_dir != repo_dir.parent:
        repo_dir = repo_dir.parent

os.chdir(repo_dir)
# Make the package importable without restarting the runtime after install.
if str(repo_dir / "src") not in sys.path:
    sys.path.insert(0, str(repo_dir / "src"))
print("Working directory:", Path.cwd())

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np
import torch

from tokenverse.vision.image_utils import (
    make_sample_image,
    pil_to_tensor,
    select_device,
)
from tokenverse.vision.mini_vit import MiniViT, ViTConfig
from tokenverse.vision.patch_embedding import PatchEmbeddingWithCLS, patchify
from tokenverse.visualization import apply_dark_theme
from tokenverse.visualization import vision_visualization as vv

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
apply_dark_theme()
%matplotlib inline
print("torch", torch.__version__, "| device:", select_device())

## 1. What a computer sees

We see objects, colours and edges. A computer starts with something considerably less romantic: a grid of numbers. A colour image has three channels (red, green and blue), and PyTorch stores it channel-first as `[C, H, W]`. Here the pixel values are scaled to `[0, 1]`.

In [ ]:
image = make_sample_image(224)  # a small drawn scene, no download needed
x = pil_to_tensor(image)  # [3, 224, 224], values in [0, 1]
print("shape:", tuple(x.shape), "| min", float(x.min()), "| max", float(x.max()))
print("values in total:", x.numel())  # 224 * 224 * 3
plt.imshow(image)
plt.axis("off")
plt.show()

That's 150,528 numbers. Passing every one of them to a Transformer as its own token would make a very long sequence, and self-attention gets more expensive with the square of the sequence length.

If you'd like to use your own picture, upload it (in Colab, through the file browser on the left) and replace the first line with `from tokenverse.vision.image_utils import load_image; image = load_image("your_file.jpg", size=224)`.

## 2. Cutting the image into patches

The [Vision Transformer paper](https://arxiv.org/abs/2010.11929) took a different approach: divide the image into fixed-size patches. With 16 × 16 patches, a 224 × 224 image gives 224 / 16 = 14 patches along each side, so 14 × 14 = 196 patches. Each one holds 16 × 16 × 3 = 768 values.

In [ ]:
batch = x.unsqueeze(0).repeat(2, 1, 1, 1)  # [2, 3, 224, 224]
patches = patchify(batch, patch_size=16)
print("input  :", tuple(batch.shape))
print("patches:", tuple(patches.shape), "  # [batch, 196 patches, 16*16*3 values]")

Patches are numbered row by row: left to right across the top, then the next row down. Patch 0 is the top-left corner, 13 the top-right and 195 the bottom-right.

In [ ]:
vv.plot_patch_grid(image, 16)
plt.show()
vv.plot_patch_thumbnails(image, 16, indices=(0, 13, 39, 102, 130, 195))
plt.show()

At this stage a patch vector is just raw pixel values, flattened. Nothing learned has touched it yet. We can rebuild a patch from its vector to check nothing got scrambled:

In [ ]:
p = patches[0, 39]  # part of the sun
print("patch 39 vector length:", p.numel())
rebuilt = p.reshape(3, 16, 16)
row, col = divmod(39, 14)
assert torch.equal(rebuilt, x[:, row * 16 : (row + 1) * 16, col * 16 : (col + 1) * 16])
print(f"patch 39 is row {row}, col {col} of the grid and rebuilds exactly")

## 3. Patch embeddings

Each 768-value patch is then projected into a shared embedding dimension, 192 here, using the same learned linear map for every patch. In code this is usually written as a `Conv2d` whose kernel size and stride both equal the patch size, so the kernel visits each patch exactly once. The two versions give the same numbers:

In [ ]:
embed = PatchEmbeddingWithCLS(
    image_size=224, patch_size=16, in_channels=3, embed_dim=192
)
conv_out = embed.embed_patches(batch)  # Conv2d version
W = embed.projection.weight.reshape(192, -1)  # [192, 768]
linear_out = patches @ W.T + embed.projection.bias  # plain linear layer
print("conv version  :", tuple(conv_out.shape))
print("linear version:", tuple(linear_out.shape))
print("same values   :", torch.allclose(conv_out, linear_out, atol=1e-5))

We now have our image as a sequence of 196 patch embeddings, `[2, 196, 192]`.

This is where vision and text part ways a little. GPT-2 looks up discrete tokens in a vocabulary. A ViT has no vocabulary of image pieces: each patch is turned into a continuous vector by a learned projection. "Token" here is used in a broader sense, meaning one element of the sequence the Transformer processes.

## 4. Meet the `[CLS]` token

`[CLS]` is a single learnable vector, stored as a parameter of shape `[1, 1, D]`, which gets placed at the front of the sequence. The idea comes from BERT, and the original ViT borrowed it. The same vector is used for every image, so before the Transformer runs it knows nothing about the input.

Why add another token when we already have 196 patches? It gives the model a dedicated position to build an image-level representation for classification. In every encoder layer, self-attention lets the CLS position attend to all the patches, and because the loss is computed from that position, training pushes it to gather whatever the classifier needs. Nobody tells it what a cat or a building looks like.

In [ ]:
out = embed(batch, return_intermediates=True)
print("CLS parameter        :", tuple(embed.cls_token.shape), "(one learned vector)")
print("expanded for batch   :", tuple(out["cls_token_expanded"].shape))
print("patch embeddings     :", tuple(out["patch_embeddings"].shape))
print("sequence with CLS    :", tuple(out["sequence_with_cls"].shape))
same = torch.equal(out["cls_token_expanded"][0], out["cls_token_expanded"][1])
print("CLS identical across images before the encoder:", same)

`expand` makes a view of the same parameter for each image in the batch rather than creating a new one per image. Prepending it gives 1 + 196 = 197 elements. Every patch is still there: CLS is added, not swapped in for anything.

In [ ]:
vv.plot_cls_sequence(196)
plt.show()

## 5. Positional embeddings

If we just flatten the image into a sequence, the model loses track of where each patch came from. Self-attention on its own has no idea that patch 14 sits directly below patch 0. So a ViT adds a learned positional embedding to every element. They're added after CLS is prepended, so CLS gets position 0 and patch *k* gets position *k + 1*, which means the table has 197 rows.

In [ ]:
print("positional embeddings:", tuple(embed.pos_embedding.shape))
expected = out["sequence_with_cls"] + embed.pos_embedding
print(
    "sequence with positions:",
    tuple(out["sequence_with_positions"].shape),
    "| matches manual sum:",
    torch.allclose(out["sequence_with_positions"], expected),
)

## 6. Let's build one

`MiniViT` (in `src/tokenverse/vision/mini_vit.py`) puts the pieces together: patch embedding with CLS and positions, a stack of Transformer encoder layers, a final `LayerNorm`, and a linear classification head applied to position 0, the CLS position. The defaults match the article: 224 × 224 images, 16 × 16 patches, dimension 192, depth 4, 3 heads and 10 classes.

The model in the article keeps PyTorch's default layer settings to stay short. `MiniViT` puts the `LayerNorm` before attention in each layer (pre-norm), as the ViT paper does. The shapes are the same either way.

In [ ]:
torch.manual_seed(0)
model = MiniViT(ViTConfig()).eval()
images = torch.randn(2, 3, 224, 224)
with torch.no_grad():
    projected = model.patch_embed.projection(images)
    feats = model.forward_features(images)
    logits = model.head(feats["cls_representation"])

print(f"{'Input images':<28} {list(images.shape)}")
print(f"{'Patch projection':<28} {list(projected.shape)}")
print(f"{'Patch embeddings':<28} {list(feats['patch_embeddings'].shape)}")
print(f"{'CLS + patch sequence':<28} {list(feats['sequence_with_cls'].shape)}")
print(f"{'Transformer encoder':<28} {list(feats['encoded_sequence'].shape)}")
print(f"{'CLS representation':<28} {list(feats['cls_representation'].shape)}")
print(f"{'Classification logits':<28} {list(logits.shape)}")

The first dimension, 2, is the batch size. Each image gives 196 patch embeddings, CLS brings the sequence to 197, and every element is a 192-dimensional vector.

These logits come from an untrained, randomly initialised model, so they show the tensor flow and nothing more.

In [ ]:
before = feats["sequence_with_cls"][:, 0]
after = feats["cls_representation"]
print(
    "CLS identical across the two images before the encoder:",
    torch.equal(before[0], before[1]),
)
print("CLS identical after the encoder:", torch.allclose(after[0], after[1]))

The CLS vectors start out identical and come out different, because attention has mixed in information from each image's patches. That's really all there is to it. Nothing special happens to CLS apart from being the position the head reads from.

In [ ]:
vv.plot_shape_flow()
plt.show()
vv.plot_vit_architecture()
plt.show()

And gradients do reach the CLS parameter, which is what makes it learnable:

In [ ]:
model.train()
loss = torch.nn.functional.cross_entropy(model(images), torch.tensor([3, 7]))
loss.backward()
print("loss:", round(loss.item(), 4))
print("CLS gradient norm:", model.cls_token.grad.norm().item())
model.zero_grad()

## 7. Optional: training a tiny ViT

To see CLS actually learn something, here's a small model trained on circles, squares and triangles drawn at random sizes, positions and colours. The images are 32 × 32 with 8 × 8 patches, so the sequence is 16 patches + CLS = 17 elements. It's a toy task, so don't read much into the accuracy, but it does show the whole loop working. Eight epochs take well under a minute on a laptop CPU.

In [ ]:
from torch.utils.data import DataLoader

from tokenverse.vision.synthetic_dataset import CLASS_NAMES, SyntheticShapes

train_ds = SyntheticShapes(num_samples=1200, image_size=32, base_seed=0)
val_ds = SyntheticShapes(num_samples=240, image_size=32, base_seed=1)

fig, axes = plt.subplots(1, 6, figsize=(9, 1.8))
for ax, i in zip(axes, range(6), strict=True):
    img, label = train_ds[i]
    ax.imshow(img.permute(1, 2, 0))
    ax.set_title(CLASS_NAMES[label], fontsize=9)
    ax.axis("off")
plt.show()

In [ ]:
RUN_TRAINING = True  # set to False to skip

if RUN_TRAINING:
    torch.manual_seed(0)
    device = select_device()
    small = ViTConfig(
        image_size=32,
        patch_size=8,
        embed_dim=64,
        depth=2,
        num_heads=4,
        mlp_dim=128,
        num_classes=3,
    )
    toy = MiniViT(small).to(device)
    opt = torch.optim.AdamW(toy.parameters(), lr=1e-3, weight_decay=0.05)
    loss_fn = torch.nn.CrossEntropyLoss()
    train_loader = DataLoader(
        train_ds,
        batch_size=32,
        shuffle=True,
        generator=torch.Generator().manual_seed(0),
    )
    val_loader = DataLoader(val_ds, batch_size=128)

    def accuracy():
        toy.eval()
        correct = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                correct += (toy(xb.to(device)).argmax(1).cpu() == yb).sum().item()
        return correct / len(val_ds)

    for epoch in range(1, 9):
        toy.train()
        total = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = loss_fn(toy(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(yb)
        print(
            f"epoch {epoch}  train loss {total / len(train_ds):.3f}  val accuracy {accuracy():.3f}"
        )

The same experiment is also a script, with a metrics CSV and a training curve: `python scripts/train_toy_vit.py`.

## 8. CLS isn't the only option

CLS isn't mandatory in every Vision Transformer. Many models skip it and average the patch outputs instead (global average pooling), and the ViT paper itself reports that this works about as well when tuned. Some use attention pooling, and others, such as Swin, build up feature maps in stages rather than one flat sequence. CLS is one way to read a prediction out of the sequence, and the original ViT happens to use it.

## Things to try

1. Change `patch_size` to 32 in `PatchEmbeddingWithCLS`. What's the new sequence length? What about 8?
2. In the toy training, swap `feats["cls_representation"]` for the average of the patch outputs, `feats["encoded_sequence"][:, 1:].mean(1)`, and retrain. Does it still learn?
3. Upload a photo and look at which patches cover the subject.